## Relationships on Reddit

### Carregamento dos dados

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

PASTA = Path("../data") 
NOME_ARQUIVO_SAIDA = "relationships.csv"

In [ ]:
dfs = []
for caminho in (PASTA.glob("*.csv")):
    parte = pd.read_csv(caminho, encoding="utf-8")
    parte["fonte"] = caminho
    print(f"{caminho.name}: {parte.shape[0]} posts, {parte.shape[1]} colunas")
    dfs.append(parte)

df = pd.concat(dfs, ignore_index=True)

actuallesbians_pastyear.csv: 454 posts, 206 colunas
gayrelationship_pastyear.csv: 472 posts, 206 colunas
interracial_relationships_pastyear.csv: 464 posts, 206 colunas
relationships_top_alltime.csv: 491 posts, 206 colunas
relationship_advice.csv: 487 posts, 206 colunas
relationship_advice_pastyear.csv: 462 posts, 206 colunas


### Verificação da qualidade dos dados

In [ ]:
df.columns

Index(['AutorPost', 'Título', 'Estado/Flair', 'Link', 'Conteúdo', 'AutorCom 1',
       'Comentário 1', 'AutorCom 2', 'Comentário 2', 'AutorCom 3',
       ...
       'Comentário 96', 'AutorCom 97', 'Comentário 97', 'AutorCom 98',
       'Comentário 98', 'AutorCom 99', 'Comentário 99', 'AutorCom 100',
       'Comentário 100', 'fonte'],
      dtype='str', length=206)

In [ ]:
cols_com = [c for c in df.columns if c.startswith("Comentário")]

df["n_comentarios"] = df[cols_com].notna().sum(axis=1)

tabela = df.groupby("fonte").agg(
    posts=("Link", "count"),
    comentarios=("n_comentarios", "sum"),
    media_por_post=("n_comentarios", "mean"),
).round(1)

tabela.loc["TOTAL"] = [
    tabela["posts"].sum(),
    tabela["comentarios"].sum(),
    round(df["n_comentarios"].mean(), 1),
]
tabela

,posts,comentarios,media_por_post
fonte,,,
..\data\actuallesbians_pastyear.csv,454.0,28944.0,63.8
..\data\gayrelationship_pastyear.csv,472.0,7414.0,15.7
..\data\interracial_relationships_pastyear.csv,464.0,45509.0,98.1
..\data\relationship_advice.csv,487.0,48347.0,99.3
..\data\relationship_advice_pastyear.csv,462.0,45372.0,98.2
..\data\relationships_top_alltime.csv,491.0,46947.0,95.6
TOTAL,2830.0,222533.0,78.6


In [ ]:
nulos = pd.DataFrame({
    "nulos": df.isna().sum(),
    "%": df.isna().mean() * 100,
    "preenchidos": df.notna().sum(),
    "dtype": df.dtypes.astype(str),
})
nulos.sort_values("%", ascending=False)

,nulos,%,preenchidos,dtype
Comentário 100,932,32.932862,1898,object
AutorCom 100,932,32.932862,1898,object
Comentário 99,930,32.862191,1900,object
AutorCom 99,930,32.862191,1900,object
AutorCom 98,924,32.650177,1906,object
...,...,...,...,...
Link,0,0.000000,2830,str
Estado/Flair,0,0.000000,2830,str
Conteúdo,0,0.000000,2830,str
fonte,0,0.000000,2830,object


In [ ]:
print("Linhas totalmente repetidas:", df.duplicated().sum())